Import Python modules

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import glob

Read in counts data

In [2]:
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)

# Convert codon_site to string to avoid issues with rows with semicolon-separated codon sites
counts_df['codon_site'] = counts_df['codon_site'].astype(str)

counts_df.head()

/tmp/ipykernel_3844042/2920330880.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,branch_length,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate
0,689,A689C,A,C,HA,2,230,GAG,GCG,E,...,891,nonsynonymous,AC,H1,HA,HA_H1,1700,avian,0.524118,0.0
1,1402,A1402C,A,C,HA,1,468,AAT,CAT,N,...,9,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.005294,0.0
2,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,20,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.011765,0.0
3,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,43,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.025294,0.0
4,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,4,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.002353,0.0


Read in predicted rates from neutral model

In [3]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.132981
1,AC,HA,AAC,0.180268
2,AC,HA,AAG,0.156194
3,AC,HA,AAT,0.195275
4,AC,HA,CAA,0.113872


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [4]:
counts_cutoff = 10
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    # subset to rows with at least X actual or expected counts
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
)
actual_expected_df.to_csv('../results/actual_expected.csv', index=False)
print("Number of unique nucleotide mutations with data:", len(actual_expected_df))
actual_expected_df.head()

Number of unique nucleotide mutations with data: 129187


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,predicted_rate,expected_count
38,1442,A1442C,A,C,PB1,2,481,AAG,ACG,K,...,AC,all,PB1,PB1_all,2273,all,83.072151,0.012038,0.199218,16.549457
66,1399,A1399C,A,C,HA,1,467,ACA,CCA,T,...,AC,H3,HA,HA_H3,1700,all,75.115294,0.000000,0.180268,13.540909
71,1453,A1453C,A,C,PB1,1,485,AAT,CAT,N,...,AC,all,PB1,PB1_all,2273,all,70.265288,0.000000,0.169610,11.917730
79,1454,A1454C,A,C,PB1,2,485,AAT,ACT,N,...,AC,all,PB1,PB1_all,2273,all,70.344039,0.000000,0.249063,17.520131
136,1423,A1423C,A,C,PB1,1,475,ATC,CTC,I,...,AC,all,PB1,PB1_all,2273,all,67.749670,0.000000,0.249063,16.873969


Compute fitness effects of synonymous nucleotide mutations, aggregating data for all synonymous mutations at a given site.

In [ ]:
pseudo_count = 0.5
groupby_cols = ['host', 'subtype', 'segment', 'gene', 'site']
site_syn_fitness_df = (
    actual_expected_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
site_syn_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of sites with estimated synonymous fitness effects:", len(site_syn_fitness_df))
site_syn_fitness_df.head()

Number of sites with estimated synonymous fitness effects: 17016


,host,subtype,segment,gene,site,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,6,143,178.497057,-0.221034
1,all,H1,HA,HA,9,42,65.435990,-0.439180
2,all,H1,HA,HA,13,83,136.123322,-0.492381
3,all,H1,HA,HA,15,51,118.790502,-0.839980
4,all,H1,HA,HA,16,10,5.833349,0.505546


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [6]:
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 86691


,host,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,171.751727,-5.842104
1,all,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,20.940427,-3.758425
2,all,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,14.385296,-3.393521
3,all,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,64.216864,-4.863169
4,all,H1,HA,HA,10,C,C,C10C,synonymous,11,9.191362,0.171112


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [7]:
aa_fitness_df.groupby(['host', 'mut_class']).size()

host   mut_class    
all    nonsense          1981
       nonsynonymous    29700
       synonymous        6718
avian  nonsense           714
       nonsynonymous    12893
       synonymous        4662
human  nonsense          1642
       nonsynonymous    23062
       synonymous        5319
dtype: int64

In [8]:
aa_fitness_df[
    (aa_fitness_df['host'] == 'all') &
    (aa_fitness_df['mut_class'] == 'nonsynonymous')
].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2569
         H3         2506
         H5         1304
         H7           15
         H9          755
MP       all        1829
NA       N1         2155
         N2         2225
         N6           94
         N8           90
         N9            8
NP       all        2777
NS       all        1877
PA       all        3628
PB1      all        3989
PB2      all        3879
dtype: int64